# V0.5 — Protocol v2: Official Food.com Splits

The dataset ships leave-one-out splits: each user has 1 held-out test interaction,
and **every test recipe has zero training interactions** (100% cold items).

This is a completely different eval from protocol v1:
- v1 (temporal cut): tests next-interaction prediction on warm items → behavioral methods dominate
- v2 (official LOO): tests cold-item ranking → content methods should have the advantage

Both evals are informative. Together they tell you where each signal type earns its keep.

In [1]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from scipy.sparse import csr_matrix
import implicit

DATA_DIR = Path("../data")

## 1. Load data and official splits

In [2]:
recipes = pd.read_csv(DATA_DIR / "RAW_recipes.csv")
recipe_embeddings = np.load(DATA_DIR / "recipe_embeddings_v0.npy")
recipe_emb_matrix = recipe_embeddings.astype(np.float32)
recipe_id_to_idx = dict(zip(recipes['id'], range(len(recipes))))

train = pd.read_csv(DATA_DIR / "interactions_train.csv")
val = pd.read_csv(DATA_DIR / "interactions_validation.csv")
test = pd.read_csv(DATA_DIR / "interactions_test.csv")

print(f"Recipes: {len(recipes):,}, Embeddings: {recipe_embeddings.shape}")
print(f"Train: {len(train):,} interactions, {train['user_id'].nunique():,} users, {train['recipe_id'].nunique():,} recipes")
print(f"Val:   {len(val):,} interactions, {val['user_id'].nunique():,} users")
print(f"Test:  {len(test):,} interactions, {test['user_id'].nunique():,} users")
print(f"\nTest recipes overlapping with train: {len(set(test['recipe_id']) & set(train['recipe_id'])):,}")
print(f"Test interactions per user: always {test.groupby('user_id').size().unique()}")

Recipes: 231,637, Embeddings: (231637, 384)
Train: 698,901 interactions, 25,076 users, 160,901 recipes
Val:   7,023 interactions, 7,023 users
Test:  12,455 interactions, 12,455 users

Test recipes overlapping with train: 0
Test interactions per user: always [1]


In [3]:
# Filter to positive interactions
train_positive = train[train['rating'] >= 4].copy()
test_positive = test[test['rating'] >= 4].copy()

# Eval cohort: users with positives in train who also have a positive test item
eval_users = set(train_positive['user_id']) & set(test_positive['user_id'])
print(f"Train positive: {len(train_positive):,}")
print(f"Test positive: {len(test_positive):,}")
print(f"Eval users (positive in both): {len(eval_users):,}")

Train positive: 645,970
Test positive: 10,393
Eval users (positive in both): 10,354


## 2. Build user representations

In [4]:
RATING_WEIGHTS = {4: 1.0, 5: 2.0}

user_train_data = train_positive.groupby('user_id').agg(
    recipe_ids=('recipe_id', list),
    ratings=('rating', list),
)

# Mean embeddings
user_embeddings = {}
user_seen_recipes = {}
for uid in eval_users:
    if uid not in user_train_data.index:
        continue
    row = user_train_data.loc[uid]
    indices, weights = [], []
    for rid, rating in zip(row['recipe_ids'], row['ratings']):
        if rid in recipe_id_to_idx:
            indices.append(recipe_id_to_idx[rid])
            weights.append(RATING_WEIGHTS.get(rating, 1.0))
    if not indices:
        continue
    weights = np.array(weights)
    emb = (recipe_embeddings[indices] * weights[:, None]).sum(axis=0) / weights.sum()
    emb = emb / np.linalg.norm(emb)
    user_embeddings[uid] = emb
    user_seen_recipes[uid] = set(row['recipe_ids'])

# Max-sim: per-user liked-recipe matrices
user_liked_embs = {}
for uid in user_embeddings:
    indices = [recipe_id_to_idx[rid] for rid in user_seen_recipes[uid] if rid in recipe_id_to_idx]
    user_liked_embs[uid] = recipe_embeddings[indices].astype(np.float32)

user_ids = list(user_embeddings.keys())
print(f"Built representations for {len(user_ids):,} users")

Built representations for 10,354 users


## 3. Train ALS on official training set

In [5]:
train_user_ids_arr = train_positive['user_id'].unique()
train_recipe_ids_arr = train_positive['recipe_id'].unique()
user_id_map = {uid: i for i, uid in enumerate(train_user_ids_arr)}
item_id_map = {rid: i for i, rid in enumerate(train_recipe_ids_arr)}
item_id_reverse = {i: rid for rid, i in item_id_map.items()}

rows = train_positive['user_id'].map(user_id_map).values
cols = train_positive['recipe_id'].map(item_id_map).values
data = np.ones(len(train_positive), dtype=np.float32)
user_item_matrix = csr_matrix((data, (rows, cols)), shape=(len(train_user_ids_arr), len(train_recipe_ids_arr)))

als_model = implicit.als.AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01, random_state=42)
als_model.fit(user_item_matrix)

print(f"ALS trained on {user_item_matrix.shape} matrix, {user_item_matrix.nnz:,} non-zeros")

# Popularity counts from train
popularity = train_positive['recipe_id'].value_counts()

  0%|          | 0/15 [00:00<?, ?it/s]

ALS trained on (24846, 153630) matrix, 645,970 non-zeros


## 4. Full-catalog retrieval eval (leave-one-out)

Each test user has exactly 1 held-out positive. We retrieve top-K from the full
231K catalog (minus seen) and check if the held-out item appears.

Key difference from protocol v1: every test item is a cold item (no training interactions).
ALS has no item factors for cold items, so it can only recommend warm items → structurally
cannot retrieve any test positive. Popularity gives cold items score 0 → same problem.

In [6]:
K = 10
recipe_ids_arr = recipes['id'].values

# Ground truth: each user's single test positive
test_user_likes = test_positive.groupby('user_id')['recipe_id'].apply(set).to_dict()

def recall_at_k(recommended, relevant):
    if not relevant:
        return 0.0
    return len(set(recommended) & set(relevant)) / len(relevant)

def hit_rate(recommended, relevant):
    return 1.0 if set(recommended) & set(relevant) else 0.0

def ndcg_at_k(recommended, relevant):
    dcg = 0.0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)
    ideal_hits = min(len(relevant), len(recommended))
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_retrieval(recs_dict, label):
    recalls, hits, ndcgs = [], [], []
    for uid in recs_dict:
        relevant = test_user_likes.get(uid, set())
        if not relevant:
            continue
        recs = recs_dict[uid]
        recalls.append(recall_at_k(recs, relevant))
        hits.append(hit_rate(recs, relevant))
        ndcgs.append(ndcg_at_k(recs, relevant))
    n_hits = sum(1 for h in hits if h > 0)
    return {'method': label, 'recall': np.mean(recalls), 'hitrate': np.mean(hits),
            'ndcg': np.mean(ndcgs), 'n_hits': n_hits, 'n_users': len(recalls)}

In [7]:
# Content mean retrieval
user_emb_matrix = np.array([user_embeddings[uid] for uid in user_ids], dtype=np.float32)
BATCH = 500
mean_recs = {}

for start in range(0, len(user_ids), BATCH):
    batch_embs = user_emb_matrix[start:start + BATCH]
    sims = batch_embs @ recipe_emb_matrix.T
    for i in range(len(batch_embs)):
        uid = user_ids[start + i]
        user_sims = sims[i].copy()
        for rid in user_seen_recipes.get(uid, set()):
            if rid in recipe_id_to_idx:
                user_sims[recipe_id_to_idx[rid]] = -np.inf
        top_k = np.argpartition(-user_sims, K)[:K]
        sorted_k = top_k[np.argsort(-user_sims[top_k])]
        mean_recs[uid] = recipe_ids_arr[sorted_k].tolist()

# Content max-sim retrieval
maxsim_recs = {}
for progress, uid in enumerate(user_ids):
    liked = user_liked_embs[uid]
    sims = liked @ recipe_emb_matrix.T
    max_sims = sims.max(axis=0)
    for rid in user_seen_recipes.get(uid, set()):
        if rid in recipe_id_to_idx:
            max_sims[recipe_id_to_idx[rid]] = -np.inf
    top_k = np.argpartition(-max_sims, K)[:K]
    sorted_k = top_k[np.argsort(-max_sims[top_k])]
    maxsim_recs[uid] = recipe_ids_arr[sorted_k].tolist()
    if (progress + 1) % 2000 == 0:
        print(f"  max-sim: {progress + 1}/{len(user_ids)}")

# Popularity retrieval
top_popular = popularity.index.tolist()
pop_recs = {}
for uid in user_ids:
    seen = user_seen_recipes.get(uid, set())
    pop_recs[uid] = [rid for rid in top_popular if rid not in seen][:K]

# ALS retrieval
als_recs = {}
for uid in user_ids:
    if uid not in user_id_map:
        continue
    uidx = user_id_map[uid]
    item_indices, scores = als_model.recommend(uidx, user_item_matrix[uidx], N=K, filter_already_liked_items=True)
    als_recs[uid] = [item_id_reverse[i] for i in item_indices]

print("All retrieval done")

  max-sim: 2000/10354
  max-sim: 4000/10354
  max-sim: 6000/10354
  max-sim: 8000/10354
  max-sim: 10000/10354
All retrieval done


In [8]:
n_catalog = len(recipes)
expected_random = K / n_catalog

results = [
    evaluate_retrieval(mean_recs, 'Content mean'),
    evaluate_retrieval(maxsim_recs, 'Content max-sim'),
    evaluate_retrieval(pop_recs, 'Popularity'),
    evaluate_retrieval(als_recs, 'Implicit ALS'),
]

n_users = results[0]['n_users']
print(f"Protocol v2 — official LOO split, {n_users:,} eval users, {n_catalog:,}-item catalog, K={K}")
print(f"All test items are COLD (zero training interactions)")
print()
print(f"{'Method':<25} {'Recall@10':>10} {'HitRate@10':>12} {'nDCG@10':>10}  {'n_hits':>7}")
print(f"{'-'*68}")
print(f"{'Random (expected)':<25} {expected_random:>10.6f} {'—':>12} {'—':>10}  {'—':>7}")
for r in results:
    print(f"{r['method']:<25} {r['recall']:>10.4f} {r['hitrate']:>12.4f} {r['ndcg']:>10.4f}  {r['n_hits']:>7,}")

Protocol v2 — official LOO split, 10,354 eval users, 231,637-item catalog, K=10
All test items are COLD (zero training interactions)

Method                     Recall@10   HitRate@10    nDCG@10   n_hits
--------------------------------------------------------------------
Random (expected)           0.000043            —          —        —
Content mean                  0.0013       0.0013     0.0005       13
Content max-sim               0.0018       0.0018     0.0011       19
Popularity                    0.0000       0.0000     0.0000        0
Implicit ALS                  0.0000       0.0000     0.0000        0


## 5. Sampled-negative ranking eval

Same setup as protocol v1: rank each test positive against 100 random negatives.
But now every positive is a cold item — ALS and popularity score them at 0.

In [9]:
N_NEGATIVES = 100
rng = np.random.RandomState(42)
all_recipe_ids = set(recipes['id'].values)

eval_triples = []
for uid in user_ids:
    relevant = test_user_likes.get(uid, set())
    if not relevant:
        continue
    seen = user_seen_recipes.get(uid, set())
    exclude = seen | relevant
    neg_pool = list(all_recipe_ids - exclude)
    for pos_rid in relevant:
        if pos_rid not in recipe_id_to_idx:
            continue
        negs = rng.choice(neg_pool, size=N_NEGATIVES, replace=False).tolist()
        eval_triples.append((uid, pos_rid, negs))

print(f"Eval triples: {len(eval_triples):,}")

Eval triples: 10,354


In [10]:
def sampled_neg_eval(score_fn, label):
    aucs, hr10s = [], []
    for uid, pos_rid, neg_rids in eval_triples:
        all_rids = [pos_rid] + neg_rids
        scores = score_fn(uid, all_rids)
        if scores is None:
            continue
        pos_score = scores[0]
        neg_scores = scores[1:]
        auc = np.mean(pos_score > neg_scores) + 0.5 * np.mean(pos_score == neg_scores)
        aucs.append(auc)
        rank = 1 + np.sum(neg_scores > pos_score)
        hr10s.append(1.0 if rank <= 10 else 0.0)
    print(f"{label:<25} AUC={np.mean(aucs):.4f}  HR@10={np.mean(hr10s):.4f}  (n={len(aucs):,})")
    return {'method': label, 'auc': np.mean(aucs), 'hr10': np.mean(hr10s)}

# Scoring functions
def content_mean_score(uid, recipe_ids):
    if uid not in user_embeddings:
        return None
    user_emb = user_embeddings[uid].astype(np.float32)
    indices = [recipe_id_to_idx.get(rid) for rid in recipe_ids]
    if any(i is None for i in indices):
        return None
    return recipe_emb_matrix[indices] @ user_emb

def content_maxsim_score(uid, recipe_ids):
    if uid not in user_liked_embs:
        return None
    liked = user_liked_embs[uid]
    indices = [recipe_id_to_idx.get(rid) for rid in recipe_ids]
    if any(i is None for i in indices):
        return None
    embs = recipe_emb_matrix[indices]
    return (embs @ liked.T).max(axis=1)

def popularity_score(uid, recipe_ids):
    return np.array([popularity.get(rid, 0) for rid in recipe_ids], dtype=np.float32)

def als_score(uid, recipe_ids):
    if uid not in user_id_map:
        return None
    uf = als_model.user_factors[user_id_map[uid]]
    return np.array([np.dot(uf, als_model.item_factors[item_id_map[rid]]) if rid in item_id_map else 0.0
                     for rid in recipe_ids], dtype=np.float32)

def random_score(uid, recipe_ids):
    return rng.rand(len(recipe_ids)).astype(np.float32)

In [11]:
print(f"Sampled-negative ranking (official LOO split, cold-item test set, {N_NEGATIVES} negatives)")
print(f"{'Method':<25} {'AUC':>8}  {'HR@10':>8}  {'n':>8}")
print(f"{'-'*55}")
sampled_neg_eval(random_score, 'Random')
sampled_neg_eval(popularity_score, 'Popularity')
sampled_neg_eval(content_mean_score, 'Content mean')
sampled_neg_eval(content_maxsim_score, 'Content max-sim')
sampled_neg_eval(als_score, 'Implicit ALS (CF)')

Sampled-negative ranking (official LOO split, cold-item test set, 100 negatives)
Method                         AUC     HR@10         n
-------------------------------------------------------
Random                    AUC=0.5025  HR@10=0.1003  (n=10,354)
Popularity                AUC=0.1684  HR@10=0.0000  (n=10,354)
Content mean              AUC=0.6056  HR@10=0.1906  (n=10,354)
Content max-sim           AUC=0.6238  HR@10=0.2213  (n=10,354)
Implicit ALS (CF)         AUC=0.4547  HR@10=0.0002  (n=10,354)


{'method': 'Implicit ALS (CF)',
 'auc': np.float64(0.4547015646127101),
 'hr10': np.float64(0.00019316206297083252)}

## 6. Interpretation

Fill in after running — compare protocol v1 vs v2 results.